# Chapter 19 — The Agent Said Done. Did Anything Change?

**Companion to *Applied AI*.**

A worker returns "done" and nobody opens the file. This notebook replays the
preserved five-case run — honest, lying, partial and wrong-target workers, plus
a decision whose basis moves after the effect — and asks of each one the only
question that matters:

## Question

**Does the world agree that the action occurred?**

## What this notebook does

It **reproduces** the pinned run's verdicts from
`decision-to-effect/2026-09-14-a1b562a/` (`results.json`, `analysis.json`,
the ledger): the worker's report and the runtime's independent reading, kept
in separate fields, compared against the intended bytes — never against each
other's claims.

```text
decided  ≠  requested  ≠  performed  ≠  observed
```

## Setup

Standard library only. No network, no API key, no `codeai` import.
Only bundle-relative paths are shown; override the evidence root with
`APPLIED_AI_EVIDENCE`.

In [1]:
import json
import os
from pathlib import Path

def find_evidence_dir(marker="decision-to-effect"):
    """Locate the preserved evidence. Override with APPLIED_AI_EVIDENCE."""
    env = os.environ.get("APPLIED_AI_EVIDENCE")
    if env and Path(env).expanduser().is_dir():
        return Path(env).expanduser()
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base / "evidence",
                     base / "experiments" / "applied-ai" / "evidence"):
            if (cand / marker).is_dir():
                return cand
    raise FileNotFoundError(
        "Preserved evidence not found. Set APPLIED_AI_EVIDENCE to the "
        "directory holding the Applied AI evidence bundles.")

EVIDENCE_DIR = find_evidence_dir()
BUNDLE = EVIDENCE_DIR / "decision-to-effect" / "2026-09-14-a1b562a"
print("bundle: decision-to-effect/2026-09-14-a1b562a")
results = json.loads((BUNDLE / "results.json").read_text(encoding="utf-8"))
analysis = json.loads((BUNDLE / "analysis.json").read_text(encoding="utf-8"))
cases = {c["case"]: c for c in results["cases"]}
print("cases :", sorted(cases))

bundle: decision-to-effect/2026-09-14-a1b562a
cases : ['honest', 'lying', 'partial', 'partial-retry', 'stale-basis', 'wrong-target']


## 1. Four workers, one report format

Three of the four workers report `succeeded`. The report is the actor's
answer. The runtime's answer lives in a different field — the resolver's
reading of the watched target — and the intended answer is the expected hash.
Recompute the only comparison that matters, observed vs intended, from the
preserved rows:

In [2]:
print(f"{'case':<13}{'report':<11}{'observed == intended?':<23}world agrees?")
print("-" * 62)
for name in ("honest", "lying", "partial", "wrong-target"):
    c = cases[name]
    agrees = c["observed_state_hash"] == c["expected_sha256"]
    # The notebook's own verdict, derived from the preserved hashes:
    print(f"{name:<13}{c['status']:<11}{str(agrees):<23}{'yes' if agrees else 'NO'}")
    assert agrees == analysis[name]["effect_matches_expected"]

print()
print("Three 'succeeded' reports; the world agrees with exactly one of them.")
print("The report and the reading travel separate paths — that is what makes")
print("the disagreement visible instead of silent.")

case         report     observed == intended?  world agrees?
--------------------------------------------------------------
honest       succeeded  True                   yes
lying        succeeded  False                  NO
partial      failed     False                  NO
wrong-target succeeded  False                  NO

Three 'succeeded' reports; the world agrees with exactly one of them.
The report and the reading travel separate paths — that is what makes
the disagreement visible instead of silent.


## 2. Lying is not the only way to be wrong

The lying worker never touches the target. The wrong-target worker writes the
exact intended bytes — into `other.toml`. Same report, same untouched target,
opposite stories about what happened. The fixture inventory, kept outside the
completion record, tells them apart:

In [3]:
print("wrong-target inventory (path, sha256[:16]):")
for path, digest in analysis["wrong-target-inventory"]:
    mark = " <-- intended bytes, wrong file" if digest == cases["wrong-target"]["expected_sha256"] else " (target, untouched)"
    print(f"  {path:<12}{digest[:16]}...{mark}")

assert cases["lying"]["observed_state_hash"] == cases["lying"]["actual_after_sha256"]
assert cases["wrong-target"]["observed_state_hash"] == cases["wrong-target"]["actual_after_sha256"]
print()
print("Both observations match the actual bytes: the runtime recorded what was")
print("there, not what the worker said. And 'the scope changed' would still not")
print("mean 'the intended thing changed' — a changed hash needs a check (Ch. 21),")
print("never just an observation.")

wrong-target inventory (path, sha256[:16]):
  cache.toml  629c3d9e25b62394... (target, untouched)
  other.toml  0fb8c6a8dbebf01b... <-- intended bytes, wrong file

Both observations match the actual bytes: the runtime recorded what was
there, not what the worker said. And 'the scope changed' would still not
mean 'the intended thing changed' — a changed hash needs a check (Ch. 21),
never just an observation.


## 3. Failure is not rollback

The partial worker raises `ConnectionError` mid-write. The target holds an
8-byte prefix of the intended bytes — a failed status with a real partial
effect. The same-key retry does not write again: it replays the FAILED result
with the adapter invoked exactly once.

In [4]:
p = cases["partial"]
print("partial status :", p["status"], "| error:", p["error"])
print("observed prefix:", p["observed_state_hash"][:16], "... != expected",
      p["expected_sha256"][:16], "...")
r = cases["partial-retry"]
print("retry reuses   :", r["reused_from_action_id"], "| invocations_total:",
      r.get("invocations_total"), "| effects:", analysis["partial-retry"]["effects"])
assert p["status"] == "failed" and p["observed_state_hash"] != p["expected_sha256"]
assert r["reused_from_action_id"] == "a-partial"
print()
print("A failed result can carry a partial effect. Reading failure as rollback")
print("would delete that fact; the ledger keeps both.")

partial status : failed | error: transport dropped mid-write
observed prefix: 492de90583611375 ... != expected 0fb8c6a8dbebf01b ...
retry reuses   : a-partial | invocations_total: 1 | effects: 1

A failed result can carry a partial effect. Reading failure as rollback
would delete that fact; the ledger keeps both.


## 4. The basis can move after the effect

The fifth case starts from a genuine recorded decision on a supported claim,
performs the intended write, and *then* refuting evidence lands. The decision
event stays intact while its standing moves — Chapter 18's rule, now spanning
an effect that already happened:

In [5]:
s = cases["stale-basis"]
print("status               :", s["status"])
print("world agrees         :", s["observed_state_hash"] == s["expected_sha256"])
print("standing before      :", s["decision_standing_before"])
print("standing after       :", s["decision_standing_after"])
print("decision event intact:", s["decision_event_intact"])
assert s["observed_state_hash"] == s["expected_sha256"]
assert (s["decision_standing_before"], s["decision_standing_after"]) == ("basis_intact", "basis_changed")
assert s["decision_event_intact"] is True
assert analysis["decision_basis_preserved"] and analysis["refuting_evidence_present"]
print()
print("New knowledge changes what is justified now, not what was done then.")

status               : succeeded
world agrees         : True
standing before      : basis_intact
standing after       : basis_changed
decision event intact: True

New knowledge changes what is justified now, not what was done then.


## 5. Why the report must never grade itself

A later probe of this same run found what the table above does not show. The
runtime *recorded* both readings for the lying case — and its own effect
projection still read the adapter's `succeeded` and called the effect
`OBSERVED`, next to byte-identical before/after readings. The word in the
explanation was "observation"; what it had was a reading it never used.

The repair is the three-dimension model from the chapter. Reproduce its logic
here, over the preserved rows — report × observation → conservative effect
state:

In [6]:
def effect_state(status, observed_hash, expected_hash, before_hash):
    """The chapter's conservative conclusion. Observation is compared with
    intention; the report is never evidence for the effect."""
    if observed_hash is None:
        return "REPORTED"    # the actor's claim, no corroboration
    if observed_hash != before_hash:
        return "OBSERVED"    # the watched scope changed — not necessarily rightly
    return "UNKNOWN"         # unchanged scope: effect may be elsewhere; reconcile

BEFORE = cases["lying"]["actual_after_sha256"]  # lying left the target as found
for name in ("honest", "lying", "wrong-target"):
    c = cases[name]
    print(f"{name:<13}report={c['status']:<10}effect_state={effect_state(c['status'], c['observed_state_hash'], c['expected_sha256'], BEFORE)}")

assert effect_state("succeeded", cases["honest"]["observed_state_hash"],
                    cases["honest"]["expected_sha256"], BEFORE) == "OBSERVED"
assert effect_state("succeeded", cases["lying"]["observed_state_hash"],
                    cases["lying"]["expected_sha256"], BEFORE) == "UNKNOWN"
print()
print("Reported success with an unchanged scope is UNKNOWN and goes to a person,")
print("not into the books as an observed effect. And OBSERVED still only means")
print("'the watched scope changed' — whether it changed rightly is a check's")
print("question (Chapter 21), never an observation's.")

honest       report=succeeded effect_state=OBSERVED
lying        report=succeeded effect_state=UNKNOWN
wrong-target report=succeeded effect_state=UNKNOWN

Reported success with an unchanged scope is UNKNOWN and goes to a person,
not into the books as an observed effect. And OBSERVED still only means
'the watched scope changed' — whether it changed rightly is a check's
question (Chapter 21), never an observation's.


## Interpretation

1. **Decided ≠ requested ≠ performed ≠ observed.** Each answers a different
   question. An observed state still has to be judged against what was intended.
2. **A worker's "done" is a report, not an observation.** The pinned run keeps
   the adapter's status/state report apart from the runtime's resolver reading
   in separate fields — and the notebook's verdicts come from comparing the
   reading with the intended bytes, never from trusting the report.
3. **Know what a hash covers.** The resolver here watches one target file.
   The wrong-target effect is caught only because the verifier-side inventory
   looks wider. Identical bytes do not prove nothing happened elsewhere —
   exactly why that case is `UNKNOWN`.
4. **Failure is not rollback; a new basis does not undo an effect.** The
   partial write and the stale-basis case keep both facts in the record.

## Try it yourself

1. Open `ledger.sqlite` in the bundle: find `action.requested`,
   `action.execution_started` (if present at this stage) and
   `action.completed` for `a-lying`. Which fields are the worker's, and which
   are the runtime's?
2. Change `effect_state` above to treat unchanged-scope success as `NONE`
   ("nothing happened"). Which later evidence — the wrong-target inventory —
   breaks that reading?
3. The partial retry replays FAILED. What would a naive retry-by-re-execution
   have done to the target? (Chapter 22 measures exactly this.)

*Evidence: `experiments/applied-ai/evidence/decision-to-effect/2026-09-14-a1b562a/`
(pinned five-case run, stdlib-only verifier, `results.json` + `analysis.json`
+ ledger). No network, no API key, no `codeai` import.*